Phase 1 : Designing the synthetic data foundation

In [2]:
import random
import uuid
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Any
import pandas as pd
import numpy as np

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [4]:
SCHEMA = {
    "company_identity": [
        "company_id",
        "company_name",
        "reporting_year",
        "sector",
        "subsector",
        "region",
        "country",
        "company_size",
        "reporting_maturity"
    ],
    "governance": [
        "board_esg_oversight",
        "board_climate_oversight",
        "esg_committee",
        "climate_committee",
        "management_responsible_role",
        "review_frequency",
        "executive_remuneration_linked_to_esg",
        "executive_remuneration_linked_to_climate"
    ],
    "strategy": [
        "main_sustainability_risks",
        "main_climate_risks",
        "main_sustainability_opportunities",
        "main_climate_opportunities",
        "time_horizon",
        "transition_plan_exists",
        "adaptation_plan_exists",
        "scenario_analysis_used",
        "resilience_assessment_level"
    ],
    "risk_management": [
        "sustainability_risk_process_formalized",
        "climate_risk_process_formalized",
        "integrated_into_enterprise_risk_management",
        "risk_identification_frequency",
        "risk_prioritization_method",
        "uses_external_frameworks"
    ],
    "metrics_targets": [
        "scope1_emissions_tco2e",
        "scope2_emissions_tco2e",
        "scope3_emissions_tco2e",
        "internal_carbon_price",
        "climate_capex_pct",
        "high_climate_risk_assets_pct",
        "climate_opportunity_assets_pct",
        "emissions_target_exists",
        "emissions_target_type",
        "emissions_target_base_year",
        "emissions_target_end_year",
        "emissions_target_progress_pct"
    ],
    "financial_effects": [
        "current_financial_effect_level",
        "short_term_financial_effect_level",
        "medium_term_financial_effect_level",
        "long_term_financial_effect_level",
        "affected_business_areas",
        "expected_cost_pressure",
        "expected_revenue_opportunity"
    ]
}

In [5]:
ALL_FIELDS = [field for section in SCHEMA.values() for field in section]

print(f"Total fields in schema: {len(ALL_FIELDS)}")
ALL_FIELDS[:10]

Total fields in schema: 51


['company_id',
 'company_name',
 'reporting_year',
 'sector',
 'subsector',
 'region',
 'country',
 'company_size',
 'reporting_maturity',
 'board_esg_oversight']

In [6]:
@dataclass
class Archetype:
    name: str
    description: str
    reporting_maturity: str
    governance_strength: str
    climate_ambition: str
    disclosure_quality: str
    likely_risks: List[str]
    likely_opportunities: List[str]
    expected_emissions_profile: str
    expected_financial_exposure: str
    default_values: Dict[str, Any] = field(default_factory=dict)

In [7]:
ARCHETYPES = [
    Archetype(
        name="climate_leader",
        description="High maturity company with strong climate governance, targets, and disclosure quality.",
        reporting_maturity="high",
        governance_strength="strong",
        climate_ambition="high",
        disclosure_quality="high",
        likely_risks=["transition risk", "supply chain risk", "physical risk"],
        likely_opportunities=["green products", "efficiency gains", "sustainable finance access"],
        expected_emissions_profile="managed",
        expected_financial_exposure="moderate",
        default_values={
            "board_climate_oversight": True,
            "climate_committee": True,
            "scenario_analysis_used": True,
            "transition_plan_exists": True,
            "adaptation_plan_exists": True,
            "emissions_target_exists": True,
            "internal_carbon_price": "likely"
        }
    ),
    Archetype(
        name="transition_exposed_emitter",
        description="Carbon-intensive company with high transition pressure and material emissions exposure.",
        reporting_maturity="medium",
        governance_strength="medium",
        climate_ambition="medium",
        disclosure_quality="medium",
        likely_risks=["transition risk", "carbon cost risk", "regulatory risk"],
        likely_opportunities=["process efficiency", "fuel switching", "low-carbon product lines"],
        expected_emissions_profile="high",
        expected_financial_exposure="high",
        default_values={
            "board_climate_oversight": True,
            "climate_committee": False,
            "scenario_analysis_used": True,
            "transition_plan_exists": True,
            "adaptation_plan_exists": False,
            "emissions_target_exists": True,
            "internal_carbon_price": "possible"
        }
    ),
    Archetype(
        name="physical_risk_exposed_operator",
        description="Company exposed to weather, water, or location-based climate disruption.",
        reporting_maturity="medium",
        governance_strength="medium",
        climate_ambition="medium",
        disclosure_quality="medium",
        likely_risks=["physical risk", "water stress", "supply chain disruption"],
        likely_opportunities=["adaptation investment", "resilient infrastructure"],
        expected_emissions_profile="moderate",
        expected_financial_exposure="high",
        default_values={
            "board_climate_oversight": True,
            "climate_committee": False,
            "scenario_analysis_used": False,
            "transition_plan_exists": False,
            "adaptation_plan_exists": True,
            "emissions_target_exists": True,
            "internal_carbon_price": "unlikely"
        }
    ),
    Archetype(
        name="low_maturity_reporter",
        description="Company with fragmented sustainability management and limited disclosure depth.",
        reporting_maturity="low",
        governance_strength="weak",
        climate_ambition="low",
        disclosure_quality="low",
        likely_risks=["compliance risk", "reputation risk", "data quality risk"],
        likely_opportunities=["basic governance improvement", "initial reporting improvement"],
        expected_emissions_profile="unknown",
        expected_financial_exposure="unclear",
        default_values={
            "board_climate_oversight": False,
            "climate_committee": False,
            "scenario_analysis_used": False,
            "transition_plan_exists": False,
            "adaptation_plan_exists": False,
            "emissions_target_exists": False,
            "internal_carbon_price": "unlikely"
        }
    ),
    Archetype(
        name="financial_sector_portfolio_exposed",
        description="Financial institution with lower operational emissions but high financed or portfolio-related exposure.",
        reporting_maturity="high",
        governance_strength="strong",
        climate_ambition="medium",
        disclosure_quality="high",
        likely_risks=["portfolio transition risk", "financed emissions risk", "reputation risk"],
        likely_opportunities=["green financing", "sustainable products", "portfolio reallocation"],
        expected_emissions_profile="low_operational_high_financed",
        expected_financial_exposure="high",
        default_values={
            "board_climate_oversight": True,
            "climate_committee": True,
            "scenario_analysis_used": True,
            "transition_plan_exists": True,
            "adaptation_plan_exists": False,
            "emissions_target_exists": True,
            "internal_carbon_price": "unlikely"
        }
    )
]

In [8]:
pd.DataFrame([asdict(a) for a in ARCHETYPES])[[
    "name", "reporting_maturity", "governance_strength",
    "climate_ambition", "disclosure_quality",
    "expected_emissions_profile", "expected_financial_exposure"
]]

,name,reporting_maturity,governance_strength,climate_ambition,disclosure_quality,expected_emissions_profile,expected_financial_exposure
0,climate_leader,high,strong,high,high,managed,moderate
1,transition_exposed_emitter,medium,medium,medium,medium,high,high
2,physical_risk_exposed_operator,medium,medium,medium,medium,moderate,high
3,low_maturity_reporter,low,weak,low,low,unknown,unclear
4,financial_sector_portfolio_exposed,high,strong,medium,high,low_operational_high_financed,high


In [9]:
@dataclass
class SectorRule:
    sector: str
    subsectors: List[str]
    likely_material_risks: List[str]
    likely_material_opportunities: List[str]
    emissions_profile: str
    likely_scope3_relevance: str
    climate_sensitivity: str
    typical_governance_maturity: str
    notes: str

In [10]:
SECTOR_RULES = [
    SectorRule(
        sector="energy",
        subsectors=["oil_and_gas", "power_generation", "renewables", "utilities"],
        likely_material_risks=[
            "transition risk", "physical risk", "carbon pricing risk",
            "regulatory risk", "asset stranding"
        ],
        likely_material_opportunities=[
            "renewable expansion", "grid modernization", "low-carbon transition"
        ],
        emissions_profile="high",
        likely_scope3_relevance="high",
        climate_sensitivity="high",
        typical_governance_maturity="medium",
        notes="Often highly material under IFRS S2 due to emissions, transition exposure, and capital intensity."
    ),
    SectorRule(
        sector="banking",
        subsectors=["retail_banking", "corporate_banking", "investment_banking", "asset_management"],
        likely_material_risks=[
            "portfolio transition risk", "credit risk", "reputation risk",
            "regulatory risk", "financed emissions exposure"
        ],
        likely_material_opportunities=[
            "green loans", "sustainable finance products", "portfolio reallocation"
        ],
        emissions_profile="low_operational_high_financed",
        likely_scope3_relevance="high",
        climate_sensitivity="high",
        typical_governance_maturity="high",
        notes="Operational emissions may be low, but financed emissions and client exposure are highly relevant."
    ),
    SectorRule(
        sector="insurance",
        subsectors=["non_life", "life", "reinsurance"],
        likely_material_risks=[
            "physical risk", "underwriting exposure", "investment portfolio risk",
            "claims volatility", "regulatory risk"
        ],
        likely_material_opportunities=[
            "climate risk products", "better pricing models", "resilient insurance design"
        ],
        emissions_profile="low_operational_investment_exposed",
        likely_scope3_relevance="medium",
        climate_sensitivity="high",
        typical_governance_maturity="high",
        notes="Physical climate impacts are central because they directly affect claims and underwriting."
    ),
    SectorRule(
        sector="industrial_transportation",
        subsectors=["aviation", "shipping", "rail", "logistics", "road_transport"],
        likely_material_risks=[
            "fuel cost risk", "transition risk", "regulatory risk",
            "physical disruption", "supply chain risk"
        ],
        likely_material_opportunities=[
            "fleet efficiency", "alternative fuels", "green logistics"
        ],
        emissions_profile="high",
        likely_scope3_relevance="medium",
        climate_sensitivity="high",
        typical_governance_maturity="medium",
        notes="Transition risk is important due to fuel dependence and decarbonization pressure."
    )
]

In [11]:
pd.DataFrame([asdict(s) for s in SECTOR_RULES])[[
    "sector", "emissions_profile", "likely_scope3_relevance",
    "climate_sensitivity", "typical_governance_maturity"
]]

,sector,emissions_profile,likely_scope3_relevance,climate_sensitivity,typical_governance_maturity
0,energy,high,high,high,medium
1,banking,low_operational_high_financed,high,high,high
2,insurance,low_operational_investment_exposed,medium,high,high
3,industrial_transportation,high,medium,high,medium


In [12]:
ARCHETYPE_MAP = {a.name: a for a in ARCHETYPES}
SECTOR_MAP = {s.sector: s for s in SECTOR_RULES}

In [13]:
def get_archetype(name: str) -> Archetype:
    return ARCHETYPE_MAP[name]

def get_sector_rule(sector: str) -> SectorRule:
    return SECTOR_MAP[sector]

def summarize_configuration(archetype_name: str, sector_name: str) -> Dict[str, Any]:
    archetype = get_archetype(archetype_name)
    sector = get_sector_rule(sector_name)

    return {
        "archetype": archetype.name,
        "sector": sector.sector,
        "reporting_maturity": archetype.reporting_maturity,
        "governance_strength": archetype.governance_strength,
        "climate_ambition": archetype.climate_ambition,
        "expected_emissions_profile": archetype.expected_emissions_profile,
        "sector_emissions_profile": sector.emissions_profile,
        "likely_risks": list(set(archetype.likely_risks + sector.likely_material_risks)),
        "likely_opportunities": list(set(archetype.likely_opportunities + sector.likely_material_opportunities)),
        "sector_notes": sector.notes
    }

In [14]:
example_config = summarize_configuration(
    archetype_name="financial_sector_portfolio_exposed",
    sector_name="banking"
)

example_config

{'archetype': 'financial_sector_portfolio_exposed',
 'sector': 'banking',
 'reporting_maturity': 'high',
 'governance_strength': 'strong',
 'climate_ambition': 'medium',
 'expected_emissions_profile': 'low_operational_high_financed',
 'sector_emissions_profile': 'low_operational_high_financed',
 'likely_risks': ['financed emissions exposure',
  'credit risk',
  'regulatory risk',
  'financed emissions risk',
  'reputation risk',
  'portfolio transition risk'],
 'likely_opportunities': ['green financing',
  'sustainable finance products',
  'green loans',
  'portfolio reallocation',
  'sustainable products'],
 'sector_notes': 'Operational emissions may be low, but financed emissions and client exposure are highly relevant.'}

In [15]:
def empty_company_profile() -> Dict[str, Any]:
    return {field: None for field in ALL_FIELDS}

In [16]:
profile = empty_company_profile()
list(profile.items())[:10]

[('company_id', None),
 ('company_name', None),
 ('reporting_year', None),
 ('sector', None),
 ('subsector', None),
 ('region', None),
 ('country', None),
 ('company_size', None),
 ('reporting_maturity', None),
 ('board_esg_oversight', None)]

In [48]:
def build_base_profile(
    company_name: str,
    sector: str,
    subsector: str,
    country: str,
    region: str,
    reporting_year: int,
    company_size: str,
    archetype_name: str
) -> Dict[str, Any]:
    archetype = get_archetype(archetype_name)
    sector_rule = get_sector_rule(sector)

    profile = empty_company_profile()

    profile["company_id"] = str(uuid.uuid4())[:8]
    profile["company_name"] = company_name
    profile["reporting_year"] = reporting_year
    profile["sector"] = sector
    profile["subsector"] = subsector
    profile["country"] = country
    profile["region"] = region
    profile["company_size"] = company_size
    profile["reporting_maturity"] = archetype.reporting_maturity
    profile["archetype_name"] = archetype_name

    profile["board_climate_oversight"] = archetype.default_values.get("board_climate_oversight")
    profile["climate_committee"] = archetype.default_values.get("climate_committee")
    profile["scenario_analysis_used"] = archetype.default_values.get("scenario_analysis_used")
    profile["transition_plan_exists"] = archetype.default_values.get("transition_plan_exists")
    profile["adaptation_plan_exists"] = archetype.default_values.get("adaptation_plan_exists")
    profile["emissions_target_exists"] = archetype.default_values.get("emissions_target_exists")

    profile["main_climate_risks"] = sector_rule.likely_material_risks[:3]
    profile["main_climate_opportunities"] = sector_rule.likely_material_opportunities[:2]
    profile["uses_external_frameworks"] = True

    return profile

In [18]:
test_profile = build_base_profile(
    company_name="Green Horizon Bank",
    sector="banking",
    subsector="corporate_banking",
    country="Tunisia",
    region="North Africa",
    reporting_year=2025,
    company_size="large",
    archetype_name="financial_sector_portfolio_exposed"
)

pd.Series(test_profile).dropna()

company_id                                                             1456b4f4
company_name                                                 Green Horizon Bank
reporting_year                                                             2025
sector                                                                  banking
subsector                                                     corporate_banking
region                                                             North Africa
country                                                                 Tunisia
company_size                                                              large
reporting_maturity                                                         high
board_climate_oversight                                                    True
climate_committee                                                          True
main_climate_risks            [portfolio transition risk, credit risk, reput...
main_climate_opportunities          [gre

Phase 2 : Generating realistic synthetic values

In [19]:
def pick_one(options, probs=None):
    return np.random.choice(options, p=probs)

def chance(probability: float) -> bool:
    return random.random() < probability

def sample_range(low: float, high: float, decimals: int = 2):
    return round(np.random.uniform(low, high), decimals)

def sample_int(low: int, high: int):
    return int(np.random.randint(low, high + 1))

In [20]:
MATURITY_PROBS = {
    "high": {
        "formal_process": 0.95,
        "scenario_analysis": 0.85,
        "targets": 0.95,
        "remuneration_link": 0.65
    },
    "medium": {
        "formal_process": 0.70,
        "scenario_analysis": 0.50,
        "targets": 0.70,
        "remuneration_link": 0.35
    },
    "low": {
        "formal_process": 0.30,
        "scenario_analysis": 0.10,
        "targets": 0.20,
        "remuneration_link": 0.10
    }
}

GOVERNANCE_FREQ_OPTIONS = ["monthly", "quarterly", "semi_annually", "annually"]
RISK_ID_FREQ_OPTIONS = ["monthly", "quarterly", "semi_annually", "annually"]
TIME_HORIZON_OPTIONS = ["short_term", "medium_term", "long_term", "short_medium_long_term"]
PRIORITIZATION_METHODS = [
    "qualitative_assessment",
    "risk_matrix",
    "scenario_analysis",
    "quantitative_scoring",
    "combined_approach"
]
RESILIENCE_LEVELS = ["basic", "moderate", "advanced"]
FIN_EFFECT_LEVELS = ["low", "moderate", "high"]

In [21]:
EMISSIONS_BASELINES = {
    "energy": {
        "small": {"scope1": (5000, 50000), "scope2": (1000, 10000), "scope3": (20000, 200000)},
        "medium": {"scope1": (50000, 300000), "scope2": (5000, 50000), "scope3": (100000, 800000)},
        "large": {"scope1": (300000, 3000000), "scope2": (50000, 300000), "scope3": (800000, 8000000)}
    },
    "banking": {
        "small": {"scope1": (50, 500), "scope2": (100, 1000), "scope3": (10000, 100000)},
        "medium": {"scope1": (500, 2000), "scope2": (1000, 5000), "scope3": (100000, 500000)},
        "large": {"scope1": (2000, 10000), "scope2": (5000, 30000), "scope3": (500000, 5000000)}
    },
    "insurance": {
        "small": {"scope1": (30, 300), "scope2": (50, 500), "scope3": (5000, 50000)},
        "medium": {"scope1": (300, 1000), "scope2": (500, 3000), "scope3": (50000, 200000)},
        "large": {"scope1": (1000, 5000), "scope2": (3000, 15000), "scope3": (200000, 1200000)}
    },
    "industrial_transportation": {
        "small": {"scope1": (1000, 10000), "scope2": (500, 5000), "scope3": (5000, 50000)},
        "medium": {"scope1": (10000, 80000), "scope2": (5000, 20000), "scope3": (50000, 300000)},
        "large": {"scope1": (80000, 600000), "scope2": (20000, 80000), "scope3": (300000, 2000000)}
    }
}

In [22]:
ARCHETYPE_EMISSIONS_MULTIPLIER = {
    "climate_leader": 0.75,
    "transition_exposed_emitter": 1.25,
    "physical_risk_exposed_operator": 1.0,
    "low_maturity_reporter": 1.1,
    "financial_sector_portfolio_exposed": 1.0
}

In [23]:
def generate_governance(profile: Dict[str, Any], archetype: Archetype, sector_rule: SectorRule) -> Dict[str, Any]:
    maturity = archetype.reporting_maturity
    probs = MATURITY_PROBS[maturity]

    if profile["board_esg_oversight"] is None:
        profile["board_esg_oversight"] = chance(probs["formal_process"])

    if profile["board_climate_oversight"] is None:
        profile["board_climate_oversight"] = chance(probs["formal_process"])

    profile["esg_committee"] = chance(probs["formal_process"])

    if profile["climate_committee"] is None:
        profile["climate_committee"] = chance(probs["formal_process"] - 0.1 if probs["formal_process"] > 0.1 else 0.05)

    role_options = [
        "chief_sustainability_officer",
        "risk_director",
        "cfo",
        "csr_manager",
        "executive_committee"
    ]

    if archetype.governance_strength == "strong":
        profile["management_responsible_role"] = pick_one(
            ["chief_sustainability_officer", "risk_director", "executive_committee"],
            probs=[0.45, 0.30, 0.25]
        )
    elif archetype.governance_strength == "medium":
        profile["management_responsible_role"] = pick_one(
            ["risk_director", "cfo", "csr_manager"],
            probs=[0.40, 0.35, 0.25]
        )
    else:
        profile["management_responsible_role"] = pick_one(
            ["cfo", "csr_manager"],
            probs=[0.60, 0.40]
        )

    if maturity == "high":
        profile["review_frequency"] = pick_one(GOVERNANCE_FREQ_OPTIONS, probs=[0.20, 0.50, 0.20, 0.10])
    elif maturity == "medium":
        profile["review_frequency"] = pick_one(GOVERNANCE_FREQ_OPTIONS, probs=[0.10, 0.40, 0.30, 0.20])
    else:
        profile["review_frequency"] = pick_one(GOVERNANCE_FREQ_OPTIONS, probs=[0.05, 0.20, 0.25, 0.50])

    profile["executive_remuneration_linked_to_esg"] = chance(probs["remuneration_link"])
    profile["executive_remuneration_linked_to_climate"] = chance(probs["remuneration_link"] - 0.1 if probs["remuneration_link"] > 0.1 else 0.05)

    return profile

In [24]:
def generate_strategy(profile: Dict[str, Any], archetype: Archetype, sector_rule: SectorRule) -> Dict[str, Any]:
    maturity = archetype.reporting_maturity
    probs = MATURITY_PROBS[maturity]

    if profile["main_sustainability_risks"] is None:
        profile["main_sustainability_risks"] = archetype.likely_risks[:3]

    if profile["main_climate_risks"] is None:
        profile["main_climate_risks"] = sector_rule.likely_material_risks[:3]

    profile["main_sustainability_opportunities"] = archetype.likely_opportunities[:3]

    if profile["main_climate_opportunities"] is None:
        profile["main_climate_opportunities"] = sector_rule.likely_material_opportunities[:2]

    if maturity == "high":
        profile["time_horizon"] = "short_medium_long_term"
    elif maturity == "medium":
        profile["time_horizon"] = pick_one(["medium_term", "long_term", "short_medium_long_term"], probs=[0.25, 0.20, 0.55])
    else:
        profile["time_horizon"] = pick_one(["short_term", "medium_term"], probs=[0.60, 0.40])

    if profile["transition_plan_exists"] is None:
        profile["transition_plan_exists"] = chance(probs["targets"])

    if profile["adaptation_plan_exists"] is None:
        profile["adaptation_plan_exists"] = chance(0.65 if sector_rule.climate_sensitivity == "high" else 0.35)

    if profile["scenario_analysis_used"] is None:
        profile["scenario_analysis_used"] = chance(probs["scenario_analysis"])

    if maturity == "high":
        profile["resilience_assessment_level"] = pick_one(RESILIENCE_LEVELS, probs=[0.10, 0.35, 0.55])
    elif maturity == "medium":
        profile["resilience_assessment_level"] = pick_one(RESILIENCE_LEVELS, probs=[0.30, 0.50, 0.20])
    else:
        profile["resilience_assessment_level"] = pick_one(RESILIENCE_LEVELS, probs=[0.70, 0.25, 0.05])

    return profile

In [25]:
def generate_risk_management(profile: Dict[str, Any], archetype: Archetype, sector_rule: SectorRule) -> Dict[str, Any]:
    maturity = archetype.reporting_maturity
    probs = MATURITY_PROBS[maturity]

    profile["sustainability_risk_process_formalized"] = chance(probs["formal_process"])
    profile["climate_risk_process_formalized"] = chance(probs["formal_process"])
    profile["integrated_into_enterprise_risk_management"] = chance(
        0.90 if maturity == "high" else 0.60 if maturity == "medium" else 0.25
    )

    if maturity == "high":
        profile["risk_identification_frequency"] = pick_one(RISK_ID_FREQ_OPTIONS, probs=[0.25, 0.45, 0.20, 0.10])
        profile["risk_prioritization_method"] = pick_one(PRIORITIZATION_METHODS, probs=[0.10, 0.20, 0.25, 0.15, 0.30])
    elif maturity == "medium":
        profile["risk_identification_frequency"] = pick_one(RISK_ID_FREQ_OPTIONS, probs=[0.10, 0.40, 0.30, 0.20])
        profile["risk_prioritization_method"] = pick_one(PRIORITIZATION_METHODS, probs=[0.25, 0.30, 0.15, 0.10, 0.20])
    else:
        profile["risk_identification_frequency"] = pick_one(RISK_ID_FREQ_OPTIONS, probs=[0.05, 0.15, 0.25, 0.55])
        profile["risk_prioritization_method"] = pick_one(PRIORITIZATION_METHODS, probs=[0.45, 0.35, 0.05, 0.05, 0.10])

    if profile["uses_external_frameworks"] is None:
        profile["uses_external_frameworks"] = chance(0.80 if maturity == "high" else 0.50 if maturity == "medium" else 0.20)

    return profile

In [26]:
def generate_emissions(sector: str, company_size: str, archetype_name: str):
    baseline = EMISSIONS_BASELINES[sector][company_size]
    multiplier = ARCHETYPE_EMISSIONS_MULTIPLIER[archetype_name]

    scope1 = round(sample_range(*baseline["scope1"]) * multiplier, 2)
    scope2 = round(sample_range(*baseline["scope2"]) * multiplier, 2)
    scope3 = round(sample_range(*baseline["scope3"]) * multiplier, 2)

    return scope1, scope2, scope3

In [27]:
def generate_metrics_targets(profile: Dict[str, Any], archetype: Archetype, sector_rule: SectorRule) -> Dict[str, Any]:
    sector = profile["sector"]
    company_size = profile["company_size"]

    scope1, scope2, scope3 = generate_emissions(sector, company_size, archetype.name)

    profile["scope1_emissions_tco2e"] = scope1
    profile["scope2_emissions_tco2e"] = scope2
    profile["scope3_emissions_tco2e"] = scope3

    if archetype.default_values.get("internal_carbon_price") == "likely":
        profile["internal_carbon_price"] = sample_range(20, 120, 2)
    elif archetype.default_values.get("internal_carbon_price") == "possible":
        profile["internal_carbon_price"] = sample_range(10, 70, 2) if chance(0.5) else None
    else:
        profile["internal_carbon_price"] = None

    if archetype.climate_ambition == "high":
        profile["climate_capex_pct"] = sample_range(15, 45, 2)
    elif archetype.climate_ambition == "medium":
        profile["climate_capex_pct"] = sample_range(5, 20, 2)
    else:
        profile["climate_capex_pct"] = sample_range(0, 8, 2)

    if sector_rule.climate_sensitivity == "high":
        profile["high_climate_risk_assets_pct"] = sample_range(10, 50, 2)
    else:
        profile["high_climate_risk_assets_pct"] = sample_range(0, 15, 2)

    if archetype.climate_ambition == "high":
        profile["climate_opportunity_assets_pct"] = sample_range(15, 50, 2)
    elif archetype.climate_ambition == "medium":
        profile["climate_opportunity_assets_pct"] = sample_range(5, 25, 2)
    else:
        profile["climate_opportunity_assets_pct"] = sample_range(0, 10, 2)

    if profile["emissions_target_exists"] is None:
        profile["emissions_target_exists"] = chance(MATURITY_PROBS[archetype.reporting_maturity]["targets"])

    if profile["emissions_target_exists"]:
        profile["emissions_target_type"] = pick_one(
            ["absolute", "intensity", "net_zero_aligned"],
            probs=[0.45, 0.35, 0.20] if archetype.climate_ambition != "high" else [0.35, 0.25, 0.40]
        )
        profile["emissions_target_base_year"] = sample_int(2018, 2024)
        profile["emissions_target_end_year"] = sample_int(2030, 2050)

        if archetype.climate_ambition == "high":
            profile["emissions_target_progress_pct"] = sample_range(20, 75, 2)
        elif archetype.climate_ambition == "medium":
            profile["emissions_target_progress_pct"] = sample_range(5, 45, 2)
        else:
            profile["emissions_target_progress_pct"] = sample_range(0, 20, 2)
    else:
        profile["emissions_target_type"] = None
        profile["emissions_target_base_year"] = None
        profile["emissions_target_end_year"] = None
        profile["emissions_target_progress_pct"] = None

    return profile

In [28]:
def generate_financial_effects(profile: Dict[str, Any], archetype: Archetype, sector_rule: SectorRule) -> Dict[str, Any]:
    exposure = archetype.expected_financial_exposure

    if exposure == "high":
        current_probs = [0.15, 0.35, 0.50]
        future_probs = [0.10, 0.30, 0.60]
    elif exposure == "moderate":
        current_probs = [0.30, 0.50, 0.20]
        future_probs = [0.20, 0.50, 0.30]
    else:
        current_probs = [0.45, 0.40, 0.15]
        future_probs = [0.35, 0.45, 0.20]

    profile["current_financial_effect_level"] = pick_one(FIN_EFFECT_LEVELS, probs=current_probs)
    profile["short_term_financial_effect_level"] = pick_one(FIN_EFFECT_LEVELS, probs=future_probs)
    profile["medium_term_financial_effect_level"] = pick_one(FIN_EFFECT_LEVELS, probs=future_probs)
    profile["long_term_financial_effect_level"] = pick_one(FIN_EFFECT_LEVELS, probs=[0.10, 0.35, 0.55])

    sector_to_business_areas = {
        "energy": ["operations", "capital_expenditure", "asset_values", "revenue_model"],
        "banking": ["loan_portfolio", "credit_risk", "investment_portfolio", "product_strategy"],
        "insurance": ["underwriting", "claims_cost", "investment_portfolio", "pricing_model"],
        "industrial_transportation": ["fleet_operations", "fuel_costs", "logistics_network", "asset_upgrades"]
    }

    areas = sector_to_business_areas.get(profile["sector"], ["operations", "strategy"])
    profile["affected_business_areas"] = random.sample(areas, k=min(2, len(areas)))

    if archetype.expected_financial_exposure == "high":
        profile["expected_cost_pressure"] = pick_one(["moderate", "high"], probs=[0.35, 0.65])
        profile["expected_revenue_opportunity"] = pick_one(["low", "moderate", "high"], probs=[0.20, 0.50, 0.30])
    else:
        profile["expected_cost_pressure"] = pick_one(["low", "moderate", "high"], probs=[0.40, 0.45, 0.15])
        profile["expected_revenue_opportunity"] = pick_one(["low", "moderate", "high"], probs=[0.30, 0.45, 0.25])

    return profile

In [29]:
def apply_consistency_rules(profile: Dict[str, Any]) -> Dict[str, Any]:
    # Rule 1: if no emissions target, all target fields must be null
    if not profile.get("emissions_target_exists"):
        profile["emissions_target_type"] = None
        profile["emissions_target_base_year"] = None
        profile["emissions_target_end_year"] = None
        profile["emissions_target_progress_pct"] = None

    # Rule 2: advanced resilience usually implies some scenario analysis
    if profile.get("resilience_assessment_level") == "advanced" and not profile.get("scenario_analysis_used"):
        profile["scenario_analysis_used"] = True

    # Rule 3: if climate oversight is false, climate committee should not be true
    if profile.get("board_climate_oversight") is False:
        profile["climate_committee"] = False

    # Rule 4: low maturity companies usually should not have very advanced structures
    if profile.get("reporting_maturity") == "low":
        if profile.get("resilience_assessment_level") == "advanced":
            profile["resilience_assessment_level"] = "basic"

    # Rule 5: target end year must be later than base year
    base_year = profile.get("emissions_target_base_year")
    end_year = profile.get("emissions_target_end_year")
    if base_year is not None and end_year is not None and end_year <= base_year:
        profile["emissions_target_end_year"] = base_year + sample_int(5, 20)

    return profile

In [30]:
def generate_company_record(
    company_name: str,
    sector: str,
    subsector: str,
    country: str,
    region: str,
    reporting_year: int,
    company_size: str,
    archetype_name: str
) -> Dict[str, Any]:
    archetype = get_archetype(archetype_name)
    sector_rule = get_sector_rule(sector)

    profile = build_base_profile(
        company_name=company_name,
        sector=sector,
        subsector=subsector,
        country=country,
        region=region,
        reporting_year=reporting_year,
        company_size=company_size,
        archetype_name=archetype_name
    )

    profile = generate_governance(profile, archetype, sector_rule)
    profile = generate_strategy(profile, archetype, sector_rule)
    profile = generate_risk_management(profile, archetype, sector_rule)
    profile = generate_metrics_targets(profile, archetype, sector_rule)
    profile = generate_financial_effects(profile, archetype, sector_rule)
    profile = apply_consistency_rules(profile)

    return profile

In [31]:
company = generate_company_record(
    company_name="North Africa Transition Bank",
    sector="banking",
    subsector="corporate_banking",
    country="Tunisia",
    region="North Africa",
    reporting_year=2025,
    company_size="large",
    archetype_name="financial_sector_portfolio_exposed"
)

pd.Series(company).dropna()

company_id                                                                             35f5c0f8
company_name                                                       North Africa Transition Bank
reporting_year                                                                             2025
sector                                                                                  banking
subsector                                                                     corporate_banking
region                                                                             North Africa
country                                                                                 Tunisia
company_size                                                                              large
reporting_maturity                                                                         high
board_esg_oversight                                                                        True
board_climate_oversight                 

In [32]:
sample_companies = [
    {
        "company_name": "North Africa Transition Bank",
        "sector": "banking",
        "subsector": "corporate_banking",
        "country": "Tunisia",
        "region": "North Africa",
        "reporting_year": 2025,
        "company_size": "large",
        "archetype_name": "financial_sector_portfolio_exposed"
    },
    {
        "company_name": "Med Solar Utility",
        "sector": "energy",
        "subsector": "renewables",
        "country": "Morocco",
        "region": "North Africa",
        "reporting_year": 2025,
        "company_size": "medium",
        "archetype_name": "climate_leader"
    },
    {
        "company_name": "TransMaghreb Logistics",
        "sector": "industrial_transportation",
        "subsector": "logistics",
        "country": "Tunisia",
        "region": "North Africa",
        "reporting_year": 2025,
        "company_size": "large",
        "archetype_name": "transition_exposed_emitter"
    },
    {
        "company_name": "Sahel Protection Insurance",
        "sector": "insurance",
        "subsector": "non_life",
        "country": "Tunisia",
        "region": "North Africa",
        "reporting_year": 2025,
        "company_size": "medium",
        "archetype_name": "physical_risk_exposed_operator"
    }
]

records = [generate_company_record(**c) for c in sample_companies]
df = pd.DataFrame(records)

df.head()

,company_id,company_name,reporting_year,sector,subsector,region,country,company_size,reporting_maturity,board_esg_oversight,...,emissions_target_base_year,emissions_target_end_year,emissions_target_progress_pct,current_financial_effect_level,short_term_financial_effect_level,medium_term_financial_effect_level,long_term_financial_effect_level,affected_business_areas,expected_cost_pressure,expected_revenue_opportunity
0,cc4d6926,North Africa Transition Bank,2025,banking,corporate_banking,North Africa,Tunisia,large,high,True,...,2019,2033,42.69,high,moderate,low,moderate,"[product_strategy, loan_portfolio]",moderate,moderate
1,8c38fcc3,Med Solar Utility,2025,energy,renewables,North Africa,Morocco,medium,high,True,...,2021,2043,60.00,moderate,moderate,moderate,high,"[capital_expenditure, revenue_model]",moderate,moderate
2,82b59cc6,TransMaghreb Logistics,2025,industrial_transportation,logistics,North Africa,Tunisia,large,medium,True,...,2020,2030,29.24,high,high,high,high,"[asset_upgrades, fleet_operations]",high,low
3,b4427da7,Sahel Protection Insurance,2025,insurance,non_life,North Africa,Tunisia,medium,medium,True,...,2024,2044,9.44,moderate,moderate,high,high,"[claims_cost, pricing_model]",high,moderate


In [33]:
def validate_dataset(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    for idx, row in df.iterrows():
        issues = []

        if row["emissions_target_exists"] is False:
            if pd.notna(row["emissions_target_type"]):
                issues.append("target exists inconsistency")

        if row["emissions_target_base_year"] is not None and row["emissions_target_end_year"] is not None:
            if row["emissions_target_end_year"] <= row["emissions_target_base_year"]:
                issues.append("target year inconsistency")

        if row["board_climate_oversight"] is False and row["climate_committee"] is True:
            issues.append("committee inconsistency")

        checks.append({
            "company_name": row["company_name"],
            "issues_found": len(issues),
            "issues": issues
        })

    return pd.DataFrame(checks)

In [34]:
validate_dataset(df)

,company_name,issues_found,issues
0,North Africa Transition Bank,0,[]
1,Med Solar Utility,0,[]
2,TransMaghreb Logistics,0,[]
3,Sahel Protection Insurance,0,[]


In [35]:
from copy import deepcopy

In [36]:
ARCHETYPE_YEARLY_TRENDS = {
    "climate_leader": {
        "scope1_change_pct": (-0.12, -0.03),
        "scope2_change_pct": (-0.15, -0.05),
        "scope3_change_pct": (-0.08, -0.02),
        "climate_capex_change": (1.0, 5.0),
        "target_progress_change": (3.0, 12.0)
    },
    "transition_exposed_emitter": {
        "scope1_change_pct": (-0.03, 0.04),
        "scope2_change_pct": (-0.05, 0.03),
        "scope3_change_pct": (-0.02, 0.05),
        "climate_capex_change": (0.0, 3.0),
        "target_progress_change": (1.0, 6.0)
    },
    "physical_risk_exposed_operator": {
        "scope1_change_pct": (-0.05, 0.03),
        "scope2_change_pct": (-0.05, 0.02),
        "scope3_change_pct": (-0.04, 0.03),
        "climate_capex_change": (0.0, 2.5),
        "target_progress_change": (1.0, 5.0)
    },
    "low_maturity_reporter": {
        "scope1_change_pct": (-0.01, 0.08),
        "scope2_change_pct": (-0.01, 0.06),
        "scope3_change_pct": (0.00, 0.08),
        "climate_capex_change": (-1.0, 1.5),
        "target_progress_change": (0.0, 3.0)
    },
    "financial_sector_portfolio_exposed": {
        "scope1_change_pct": (-0.06, 0.02),
        "scope2_change_pct": (-0.08, 0.01),
        "scope3_change_pct": (-0.04, 0.04),
        "climate_capex_change": (0.5, 4.0),
        "target_progress_change": (2.0, 8.0)
    }
}

In [37]:
def evolve_numeric(value, pct_low, pct_high, floor=0, decimals=2):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    pct = np.random.uniform(pct_low, pct_high)
    new_value = value * (1 + pct)
    return round(max(floor, new_value), decimals)

def evolve_absolute(value, change_low, change_high, floor=0, decimals=2):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    delta = np.random.uniform(change_low, change_high)
    new_value = value + delta
    return round(max(floor, new_value), decimals)

In [44]:
def maybe_improve_boolean(current_value, improve_prob=0.15):
    if current_value is True:
        return True
    if current_value is False and chance(improve_prob):
        return True
    return current_value

In [45]:
def evolve_company_year(previous_record: Dict[str, Any]) -> Dict[str, Any]:
    record = deepcopy(previous_record)
    archetype_name = record["archetype_name"]
    trend = ARCHETYPE_YEARLY_TRENDS[archetype_name]

    record["reporting_year"] += 1

    # emissions
    record["scope1_emissions_tco2e"] = evolve_numeric(
        record["scope1_emissions_tco2e"], *trend["scope1_change_pct"]
    )
    record["scope2_emissions_tco2e"] = evolve_numeric(
        record["scope2_emissions_tco2e"], *trend["scope2_change_pct"]
    )
    record["scope3_emissions_tco2e"] = evolve_numeric(
        record["scope3_emissions_tco2e"], *trend["scope3_change_pct"]
    )

    # capex and opportunity exposure
    record["climate_capex_pct"] = evolve_absolute(
        record["climate_capex_pct"], *trend["climate_capex_change"], floor=0
    )
    record["climate_opportunity_assets_pct"] = evolve_absolute(
        record["climate_opportunity_assets_pct"], -1.0, 3.0, floor=0
    )
    record["high_climate_risk_assets_pct"] = evolve_absolute(
        record["high_climate_risk_assets_pct"], -2.0, 2.0, floor=0
    )

    # governance and process can improve gradually
    record["board_esg_oversight"] = maybe_improve_boolean(record["board_esg_oversight"], 0.08)
    record["board_climate_oversight"] = maybe_improve_boolean(record["board_climate_oversight"], 0.10)
    record["esg_committee"] = maybe_improve_boolean(record["esg_committee"], 0.12)
    record["climate_committee"] = maybe_improve_boolean(record["climate_committee"], 0.12)
    record["scenario_analysis_used"] = maybe_improve_boolean(record["scenario_analysis_used"], 0.15)
    record["transition_plan_exists"] = maybe_improve_boolean(record["transition_plan_exists"], 0.12)
    record["adaptation_plan_exists"] = maybe_improve_boolean(record["adaptation_plan_exists"], 0.10)
    record["sustainability_risk_process_formalized"] = maybe_improve_boolean(
        record["sustainability_risk_process_formalized"], 0.12
    )
    record["climate_risk_process_formalized"] = maybe_improve_boolean(
        record["climate_risk_process_formalized"], 0.12
    )
    record["integrated_into_enterprise_risk_management"] = maybe_improve_boolean(
        record["integrated_into_enterprise_risk_management"], 0.12
    )

    # target progress
    if record.get("emissions_target_exists") and record.get("emissions_target_progress_pct") is not None:
        record["emissions_target_progress_pct"] = evolve_absolute(
            record["emissions_target_progress_pct"],
            *trend["target_progress_change"],
            floor=0
        )
        record["emissions_target_progress_pct"] = min(record["emissions_target_progress_pct"], 100.0)

    # internal carbon price evolves if exists
    if record.get("internal_carbon_price") is not None:
        record["internal_carbon_price"] = evolve_absolute(
            record["internal_carbon_price"], -3.0, 8.0, floor=0
        )

    # financial effect logic: long-term pressure can intensify
    fin_order = {"low": 0, "moderate": 1, "high": 2}
    rev_fin_order = {0: "low", 1: "moderate", 2: "high"}

    for col in [
        "current_financial_effect_level",
        "short_term_financial_effect_level",
        "medium_term_financial_effect_level",
        "long_term_financial_effect_level"
    ]:
        current_val = fin_order[record[col]]
        shift = np.random.choice([-1, 0, 1], p=[0.10, 0.60, 0.30])
        if col == "long_term_financial_effect_level":
            shift = np.random.choice([0, 1], p=[0.55, 0.45])
        new_val = min(2, max(0, current_val + shift))
        record[col] = rev_fin_order[new_val]

    record = apply_consistency_rules(record)
    return record

In [46]:
def generate_company_timeseries(
    company_name: str,
    sector: str,
    subsector: str,
    country: str,
    region: str,
    start_year: int,
    end_year: int,
    company_size: str,
    archetype_name: str
) -> List[Dict[str, Any]]:
    records = []

    first = generate_company_record(
        company_name=company_name,
        sector=sector,
        subsector=subsector,
        country=country,
        region=region,
        reporting_year=start_year,
        company_size=company_size,
        archetype_name=archetype_name
    )
    records.append(first)

    current = first
    for _ in range(start_year + 1, end_year + 1):
        current = evolve_company_year(current)
        records.append(current)

    return records

In [49]:
ts_records = generate_company_timeseries(
    company_name="North Africa Transition Bank",
    sector="banking",
    subsector="corporate_banking",
    country="Tunisia",
    region="North Africa",
    start_year=2022,
    end_year=2025,
    company_size="large",
    archetype_name="financial_sector_portfolio_exposed"
)

ts_df = pd.DataFrame(ts_records)
ts_df[[
    "company_name",
    "reporting_year",
    "scope1_emissions_tco2e",
    "scope2_emissions_tco2e",
    "scope3_emissions_tco2e",
    "climate_capex_pct",
    "emissions_target_progress_pct"
]]

,company_name,reporting_year,scope1_emissions_tco2e,scope2_emissions_tco2e,scope3_emissions_tco2e,climate_capex_pct,emissions_target_progress_pct
0,North Africa Transition Bank,2022,2132.70,17802.33,1519230.99,14.68,10.50
1,North Africa Transition Bank,2023,2144.17,17267.69,1522834.68,16.03,17.90
2,North Africa Transition Bank,2024,2167.68,17098.27,1540137.94,16.82,23.54
3,North Africa Transition Bank,2025,2132.78,16795.13,1558861.24,18.10,27.49


In [50]:
MISSINGNESS_BY_MATURITY = {
    "high": {
        "quantitative": 0.05,
        "qualitative": 0.02,
        "advanced_climate": 0.08
    },
    "medium": {
        "quantitative": 0.15,
        "qualitative": 0.08,
        "advanced_climate": 0.22
    },
    "low": {
        "quantitative": 0.35,
        "qualitative": 0.18,
        "advanced_climate": 0.50
    }
}

In [51]:
QUANT_FIELDS = [
    "scope1_emissions_tco2e",
    "scope2_emissions_tco2e",
    "scope3_emissions_tco2e",
    "climate_capex_pct",
    "high_climate_risk_assets_pct",
    "climate_opportunity_assets_pct",
    "internal_carbon_price",
    "emissions_target_progress_pct"
]

QUAL_FIELDS = [
    "management_responsible_role",
    "review_frequency",
    "risk_identification_frequency",
    "risk_prioritization_method",
    "time_horizon"
]

ADVANCED_CLIMATE_FIELDS = [
    "scenario_analysis_used",
    "transition_plan_exists",
    "adaptation_plan_exists",
    "resilience_assessment_level",
    "internal_carbon_price"
]

In [52]:
def apply_missingness(record: Dict[str, Any]) -> Dict[str, Any]:
    maturity = record["reporting_maturity"]
    miss = MISSINGNESS_BY_MATURITY[maturity]
    record = deepcopy(record)

    for field in QUANT_FIELDS:
        if field in record and chance(miss["quantitative"]):
            record[field] = None

    for field in QUAL_FIELDS:
        if field in record and chance(miss["qualitative"]):
            record[field] = None

    for field in ADVANCED_CLIMATE_FIELDS:
        if field in record and chance(miss["advanced_climate"]):
            record[field] = None

    # keep core identifiers safe
    protected = {
        "company_name", "sector", "subsector", "country", "region",
        "reporting_year", "company_size", "archetype_name", "reporting_maturity"
    }
    for p in protected:
        pass

    return apply_consistency_rules(record)

In [53]:
def generate_energy_sector_metrics(record: Dict[str, Any]) -> Dict[str, Any]:
    record = deepcopy(record)

    if record["sector"] != "energy":
        return record

    record["renewable_generation_share_pct"] = sample_range(5, 95, 2)
    record["fossil_generation_share_pct"] = round(max(0, 100 - record["renewable_generation_share_pct"]), 2)
    record["installed_clean_capacity_mw"] = sample_range(50, 5000, 2)
    record["grid_exposure_level"] = pick_one(["low", "moderate", "high"], probs=[0.25, 0.45, 0.30])

    return record

In [54]:
def generate_transport_industry_metrics(record: Dict[str, Any]) -> Dict[str, Any]:
    record = deepcopy(record)

    if record["sector"] != "industrial_transportation":
        return record

    record["fuel_intensity_index"] = sample_range(70, 140, 2)
    record["low_emission_fleet_share_pct"] = sample_range(0, 60, 2)
    record["logistics_network_climate_exposure_pct"] = sample_range(5, 50, 2)
    record["asset_retrofit_needs_pct"] = sample_range(5, 70, 2)

    return record

In [55]:
def generate_banking_metrics(record: Dict[str, Any]) -> Dict[str, Any]:
    record = deepcopy(record)

    if record["sector"] != "banking":
        return record

    size = record["company_size"]

    if size == "small":
        portfolio = sample_range(200, 1500, 2)
    elif size == "medium":
        portfolio = sample_range(1500, 10000, 2)
    else:
        portfolio = sample_range(10000, 80000, 2)

    record["gross_loan_portfolio_musd"] = portfolio
    record["carbon_intensive_portfolio_share_pct"] = sample_range(10, 55, 2)
    record["green_finance_share_pct"] = sample_range(2, 35, 2)
    record["financed_emissions_tco2e"] = round(
        portfolio * sample_range(20, 200, 2), 2
    )
    record["climate_sensitive_collateral_share_pct"] = sample_range(5, 40, 2)
    record["portfolio_under_transition_plan_pct"] = sample_range(0, 45, 2)

    return record

In [56]:
def generate_insurance_metrics(record: Dict[str, Any]) -> Dict[str, Any]:
    record = deepcopy(record)

    if record["sector"] != "insurance":
        return record

    record["cat_exposed_policies_share_pct"] = sample_range(5, 60, 2)
    record["climate_related_claims_ratio_pct"] = sample_range(1, 30, 2)
    record["insured_asset_high_risk_share_pct"] = sample_range(5, 50, 2)
    record["green_insurance_products_share_pct"] = sample_range(0, 25, 2)
    record["reinsurance_dependence_pct"] = sample_range(10, 70, 2)

    return record

In [57]:
def add_sector_specific_metrics(record: Dict[str, Any]) -> Dict[str, Any]:
    record = generate_energy_sector_metrics(record)
    record = generate_transport_industry_metrics(record)
    record = generate_banking_metrics(record)
    record = generate_insurance_metrics(record)
    return record

In [62]:
def enrich_target_logic(record: Dict[str, Any]) -> Dict[str, Any]:
    record = deepcopy(record)

    if not record.get("emissions_target_exists"):
        record["target_validation_status"] = None
        record["target_scope_coverage"] = None
        record["target_horizon_category"] = None
        return record

    maturity = record.get("reporting_maturity", "medium")
    ambition = record.get("climate_ambition", "medium")

    if maturity == "high":
        record["target_validation_status"] = pick_one(
            ["internally_defined", "externally_aligned", "science_based_style"],
            probs=[0.25, 0.35, 0.40]
        )
    elif maturity == "medium":
        record["target_validation_status"] = pick_one(
            ["internally_defined", "externally_aligned"],
            probs=[0.65, 0.35]
        )
    else:
        record["target_validation_status"] = "internally_defined"

    if ambition == "high":
        record["target_scope_coverage"] = pick_one(
            ["scope1_2", "scope1_2_3"],
            probs=[0.35, 0.65]
        )
    else:
        record["target_scope_coverage"] = pick_one(
            ["scope1_2", "scope1_2_3"],
            probs=[0.70, 0.30]
        )

    end_year = record.get("emissions_target_end_year")
    reporting_year = record.get("reporting_year")

    if end_year is not None and reporting_year is not None:
        diff = end_year - reporting_year
        if diff <= 3:
            record["target_horizon_category"] = "short_term"
        elif diff <= 10:
            record["target_horizon_category"] = "medium_term"
        else:
            record["target_horizon_category"] = "long_term"
    else:
        record["target_horizon_category"] = None

    return record

In [63]:
def generate_company_record_v2(
    company_name: str,
    sector: str,
    subsector: str,
    country: str,
    region: str,
    reporting_year: int,
    company_size: str,
    archetype_name: str,
    apply_missing_data: bool = True
) -> Dict[str, Any]:
    record = generate_company_record(
        company_name=company_name,
        sector=sector,
        subsector=subsector,
        country=country,
        region=region,
        reporting_year=reporting_year,
        company_size=company_size,
        archetype_name=archetype_name
    )

    record = add_sector_specific_metrics(record)
    record = enrich_target_logic(record)

    if apply_missing_data:
        record = apply_missingness(record)

    record = apply_consistency_rules(record)
    return record

In [64]:
def generate_company_timeseries_v2(
    company_name: str,
    sector: str,
    subsector: str,
    country: str,
    region: str,
    start_year: int,
    end_year: int,
    company_size: str,
    archetype_name: str,
    apply_missing_data: bool = True
) -> List[Dict[str, Any]]:
    records = []

    first = generate_company_record_v2(
        company_name=company_name,
        sector=sector,
        subsector=subsector,
        country=country,
        region=region,
        reporting_year=start_year,
        company_size=company_size,
        archetype_name=archetype_name,
        apply_missing_data=apply_missing_data
    )
    records.append(first)

    current = deepcopy(first)
    for _ in range(start_year + 1, end_year + 1):
        current = evolve_company_year(current)
        current = add_sector_specific_metrics(current)
        current = enrich_target_logic(current)

        if apply_missing_data:
            current = apply_missingness(current)

        current = apply_consistency_rules(current)
        records.append(deepcopy(current))

    return records

In [65]:
company_specs = [
    {
        "company_name": "North Africa Transition Bank",
        "sector": "banking",
        "subsector": "corporate_banking",
        "country": "Tunisia",
        "region": "North Africa",
        "start_year": 2022,
        "end_year": 2025,
        "company_size": "large",
        "archetype_name": "financial_sector_portfolio_exposed"
    },
    {
        "company_name": "Med Solar Utility",
        "sector": "energy",
        "subsector": "renewables",
        "country": "Morocco",
        "region": "North Africa",
        "start_year": 2022,
        "end_year": 2025,
        "company_size": "medium",
        "archetype_name": "climate_leader"
    },
    {
        "company_name": "TransMaghreb Logistics",
        "sector": "industrial_transportation",
        "subsector": "logistics",
        "country": "Tunisia",
        "region": "North Africa",
        "start_year": 2022,
        "end_year": 2025,
        "company_size": "large",
        "archetype_name": "transition_exposed_emitter"
    },
    {
        "company_name": "Sahel Protection Insurance",
        "sector": "insurance",
        "subsector": "non_life",
        "country": "Tunisia",
        "region": "North Africa",
        "start_year": 2022,
        "end_year": 2025,
        "company_size": "medium",
        "archetype_name": "physical_risk_exposed_operator"
    },
    {
        "company_name": "Coastal Industrial Group",
        "sector": "industrial_transportation",
        "subsector": "manufacturing_logistics",
        "country": "Algeria",
        "region": "North Africa",
        "start_year": 2022,
        "end_year": 2025,
        "company_size": "medium",
        "archetype_name": "low_maturity_reporter"
    }
]

all_records = []
for spec in company_specs:
    series = generate_company_timeseries_v2(**spec, apply_missing_data=True)
    all_records.extend(series)

df_v2 = pd.DataFrame(all_records)
df_v2.head()

,company_id,company_name,reporting_year,sector,subsector,region,country,company_size,reporting_maturity,board_esg_oversight,...,grid_exposure_level,fuel_intensity_index,low_emission_fleet_share_pct,logistics_network_climate_exposure_pct,asset_retrofit_needs_pct,cat_exposed_policies_share_pct,climate_related_claims_ratio_pct,insured_asset_high_risk_share_pct,green_insurance_products_share_pct,reinsurance_dependence_pct
0,f6b3026d,North Africa Transition Bank,2022,banking,corporate_banking,North Africa,Tunisia,large,high,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,f6b3026d,North Africa Transition Bank,2023,banking,corporate_banking,North Africa,Tunisia,large,high,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f6b3026d,North Africa Transition Bank,2024,banking,corporate_banking,North Africa,Tunisia,large,high,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,f6b3026d,North Africa Transition Bank,2025,banking,corporate_banking,North Africa,Tunisia,large,high,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,31285669,Med Solar Utility,2022,energy,renewables,North Africa,Morocco,medium,high,True,...,low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
def validate_dataset_v2(df: pd.DataFrame) -> pd.DataFrame:
    results = []

    for idx, row in df.iterrows():
        issues = []

        if row.get("emissions_target_exists") is False:
            for field in [
                "emissions_target_type",
                "emissions_target_base_year",
                "emissions_target_end_year",
                "emissions_target_progress_pct",
                "target_validation_status",
                "target_scope_coverage",
                "target_horizon_category"
            ]:
                if pd.notna(row.get(field)):
                    issues.append(f"{field} should be null when no target exists")

        if pd.notna(row.get("emissions_target_base_year")) and pd.notna(row.get("emissions_target_end_year")):
            if row["emissions_target_end_year"] <= row["emissions_target_base_year"]:
                issues.append("target end year not greater than base year")

        if row.get("board_climate_oversight") is False and row.get("climate_committee") is True:
            issues.append("climate committee without climate oversight")

        for pct_col in [
            "climate_capex_pct",
            "high_climate_risk_assets_pct",
            "climate_opportunity_assets_pct",
            "emissions_target_progress_pct",
            "green_finance_share_pct",
            "carbon_intensive_portfolio_share_pct",
            "portfolio_under_transition_plan_pct",
            "renewable_generation_share_pct",
            "fossil_generation_share_pct",
            "low_emission_fleet_share_pct",
            "cat_exposed_policies_share_pct"
        ]:
            val = row.get(pct_col)
            if pd.notna(val) and (val < 0 or val > 100):
                issues.append(f"{pct_col} outside 0-100")

        results.append({
            "company_name": row["company_name"],
            "reporting_year": row["reporting_year"],
            "issues_found": len(issues),
            "issues": issues
        })

    return pd.DataFrame(results)

In [67]:
validation_results = validate_dataset_v2(df_v2)
validation_results.head(10)

,company_name,reporting_year,issues_found,issues
0,North Africa Transition Bank,2022,0,[]
1,North Africa Transition Bank,2023,0,[]
2,North Africa Transition Bank,2024,0,[]
3,North Africa Transition Bank,2025,0,[]
4,Med Solar Utility,2022,0,[]
5,Med Solar Utility,2023,0,[]
6,Med Solar Utility,2024,0,[]
7,Med Solar Utility,2025,0,[]
8,TransMaghreb Logistics,2022,0,[]
9,TransMaghreb Logistics,2023,0,[]


In [68]:
validation_results["issues_found"].value_counts(dropna=False)

issues_found
0    20
Name: count, dtype: int64

In [69]:
df_v2.to_csv("synthetic_ifrs_dataset_v2_realism.csv", index=False)
print("Saved synthetic_ifrs_dataset_v2_realism.csv")

Saved synthetic_ifrs_dataset_v2_realism.csv


In [70]:
def pick_one(options, probs=None):
    return np.random.choice(options, p=probs) if probs is not None else random.choice(options)

def maybe(value, prob=0.8):
    return value if random.random() < prob else np.nan

def clamp(x, low, high):
    return max(low, min(high, x))

def safe_float(x, default=0.0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except:
        return default

In [71]:
SCOPE3_CATEGORY_COLUMNS = [
    "scope3_cat_1_purchased_goods_services_tco2e",
    "scope3_cat_2_capital_goods_tco2e",
    "scope3_cat_3_fuel_energy_related_tco2e",
    "scope3_cat_4_upstream_transport_tco2e",
    "scope3_cat_5_waste_tco2e",
    "scope3_cat_6_business_travel_tco2e",
    "scope3_cat_7_employee_commuting_tco2e",
    "scope3_cat_8_upstream_leased_assets_tco2e",
    "scope3_cat_9_downstream_transport_tco2e",
    "scope3_cat_10_processing_sold_products_tco2e",
    "scope3_cat_11_use_of_sold_products_tco2e",
    "scope3_cat_12_end_of_life_tco2e",
    "scope3_cat_13_downstream_leased_assets_tco2e",
    "scope3_cat_14_franchises_tco2e",
    "scope3_cat_15_investments_tco2e"
]

SCOPE3_PROFILES = {
    "Energy": np.array([0.22, 0.10, 0.08, 0.08, 0.02, 0.01, 0.01, 0.01, 0.06, 0.04, 0.30, 0.05, 0.00, 0.00, 0.02]),
    "Banking": np.array([0.02, 0.01, 0.01, 0.01, 0.01, 0.02, 0.01, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.91]),
    "Insurance": np.array([0.03, 0.01, 0.01, 0.01, 0.01, 0.02, 0.01, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.91]),
    "Industrial Transportation": np.array([0.18, 0.08, 0.10, 0.14, 0.02, 0.02, 0.01, 0.01, 0.10, 0.03, 0.22, 0.06, 0.01, 0.00, 0.02]),
    "Default": np.array([0.25, 0.08, 0.07, 0.10, 0.03, 0.03, 0.02, 0.01, 0.06, 0.03, 0.20, 0.08, 0.01, 0.01, 0.02])
}

for k in SCOPE3_PROFILES:
    SCOPE3_PROFILES[k] = SCOPE3_PROFILES[k] / SCOPE3_PROFILES[k].sum()

In [72]:
def enrich_ifrs_s2_disclosures(row):
    sector = row.get("sector", "Default")
    maturity = row.get("reporting_maturity", "medium")
    company_size = row.get("company_size", "medium")
    
    # -----------------------------
    # 1) Scope 1, 2, 3 completion
    # -----------------------------
    revenue = safe_float(row.get("revenue_musd", np.nan), default=500.0)

    if pd.isna(row.get("scope1_emissions_tco2e", np.nan)):
        sector_multiplier = {
            "Energy": 220,
            "Industrial Transportation": 140,
            "Insurance": 8,
            "Banking": 6
        }.get(sector, 40)
        
        size_factor = {
            "small": 0.5,
            "medium": 1.0,
            "large": 2.0
        }.get(company_size, 1.0)
        
        row["scope1_emissions_tco2e"] = round(revenue * sector_multiplier * size_factor * np.random.uniform(0.6, 1.4), 2)

    if pd.isna(row.get("scope2_emissions_tco2e", np.nan)):
        scope1 = safe_float(row["scope1_emissions_tco2e"])
        row["scope2_emissions_tco2e"] = round(scope1 * np.random.uniform(0.18, 0.55), 2)

    if pd.isna(row.get("scope3_emissions_tco2e", np.nan)):
        scope1 = safe_float(row["scope1_emissions_tco2e"])
        scope2 = safe_float(row["scope2_emissions_tco2e"])
        
        sector_factor = {
            "Energy": np.random.uniform(3.5, 8.0),
            "Industrial Transportation": np.random.uniform(2.5, 6.0),
            "Insurance": np.random.uniform(8.0, 20.0),
            "Banking": np.random.uniform(15.0, 40.0)
        }.get(sector, np.random.uniform(3.0, 8.0))
        
        row["scope3_emissions_tco2e"] = round((scope1 + scope2) * sector_factor, 2)

    # -----------------------------
    # 2) Scope 3 category breakdown
    # -----------------------------
    total_scope3 = safe_float(row["scope3_emissions_tco2e"])
    profile = SCOPE3_PROFILES.get(sector, SCOPE3_PROFILES["Default"])
    
    noise = np.random.uniform(0.85, 1.15, size=len(profile))
    adjusted = profile * noise
    adjusted = adjusted / adjusted.sum()
    
    category_values = (adjusted * total_scope3).round(2)
    
    # fix rounding drift
    drift = round(total_scope3 - category_values.sum(), 2)
    category_values[0] += drift
    
    for col, val in zip(SCOPE3_CATEGORY_COLUMNS, category_values):
        row[col] = val

    row["scope3_category_coverage_pct"] = round(np.random.uniform(72, 98), 1) if maturity != "low" else round(np.random.uniform(45, 75), 1)

    # -----------------------------
    # 3) GHG methodology fields
    # -----------------------------
    row["ghg_measurement_standard"] = "GHG Protocol Corporate Standard"
    row["organizational_boundary_approach"] = pick_one(
        ["equity share", "financial control", "operational control"],
        probs=[0.15, 0.20, 0.65]
    )
    row["operational_boundary_approach"] = "Scope 1, Scope 2 and relevant Scope 3 categories"
    row["scope2_accounting_approach"] = pick_one(
        ["location-based only", "market-based and location-based"],
        probs=[0.35, 0.65]
    )
    row["emission_factor_source"] = pick_one([
        "IEA emission factors",
        "DEFRA/BEIS emission factors",
        "EPA emission factors",
        "national grid emission factors",
        "supplier-specific factors where available"
    ])
    row["base_year"] = pick_one([2019, 2020, 2021, 2022])
    row["base_year_recalculation_policy"] = pick_one([
        "recalculated for structural changes above threshold",
        "recalculated for acquisitions/divestments and methodology changes",
        "no recalculation unless material boundary change"
    ])
    row["data_coverage_pct"] = round(np.random.uniform(75, 99), 1) if maturity != "low" else round(np.random.uniform(55, 85), 1)
    row["third_party_assurance_level"] = pick_one(
        ["none", "limited", "reasonable"],
        probs=[0.35, 0.50, 0.15] if maturity == "high" else [0.55, 0.40, 0.05]
    )

    row["scope1_methodology_note"] = "Calculated from direct fuel combustion and owned operational sources using activity data and published emission factors."
    row["scope2_methodology_note"] = "Calculated from purchased electricity consumption using location-based factors and, where available, market-based supplier factors."
    row["scope3_methodology_note"] = "Estimated for relevant upstream and downstream categories using spend-based, activity-based and supplier-specific methods where available."

    # -----------------------------
    # 4) Internal carbon price
    # -----------------------------
    ambition = str(row.get("climate_ambition", "medium")).lower()
    
    use_icp = (
        maturity in ["medium", "high"]
        and ambition in ["medium", "high", "very_high"]
        and random.random() < 0.7
    )
    
    row["internal_carbon_price_used"] = use_icp
    
    if use_icp:
        row["internal_carbon_price"] = round(np.random.uniform(25, 110), 2)
        row["internal_carbon_price_currency"] = "USD/tCO2e"
        row["internal_carbon_price_application_area"] = pick_one([
            "capital allocation and project appraisal",
            "strategic planning and investment decisions",
            "risk assessment and portfolio steering",
            "procurement and operational efficiency decisions"
        ])
    else:
        row["internal_carbon_price"] = np.nan
        row["internal_carbon_price_currency"] = np.nan
        row["internal_carbon_price_application_area"] = np.nan

    # -----------------------------
    # 5) Risk management methodology
    # -----------------------------
    row["climate_risk_identification_process"] = pick_one([
        "bottom-up business unit assessment with central review",
        "enterprise risk management process integrated with sustainability review",
        "top-down screening followed by asset-level validation"
    ])
    row["climate_risk_assessment_method"] = pick_one([
        "qualitative screening plus quantitative scoring",
        "inherent and residual risk scoring matrix",
        "exposure-sensitivity-adaptive capacity assessment"
    ])
    row["climate_risk_likelihood_method"] = pick_one([
        "expert judgment with defined scoring criteria",
        "historical trends and forward-looking indicators",
        "probability bands aligned with enterprise risk framework"
    ])
    row["climate_risk_magnitude_method"] = pick_one([
        "estimated financial impact bands",
        "operational disruption and financial impact scoring",
        "revenue-cost-asset exposure severity scoring"
    ])
    row["climate_risk_prioritization_basis"] = pick_one([
        "likelihood x magnitude",
        "financial materiality and time horizon",
        "exposure severity and management readiness"
    ])
    row["climate_risk_review_frequency"] = pick_one(["annual", "semi-annual", "quarterly"], probs=[0.6, 0.3, 0.1])
    row["integration_into_enterprise_risk_management"] = pick_one(
        ["fully integrated", "partially integrated", "standalone climate process"],
        probs=[0.45, 0.40, 0.15]
    )

    # -----------------------------
    # 6) Transition plan schema
    # -----------------------------
    has_transition_plan = random.random() < (0.8 if maturity == "high" else 0.55 if maturity == "medium" else 0.25)
    row["transition_plan_exists"] = has_transition_plan
    
    if has_transition_plan:
        row["transition_plan_horizon"] = pick_one(["to 2030", "to 2035", "to 2050 with interim targets"])
        row["transition_plan_coverage_scope"] = pick_one([
            "operations only",
            "operations and value chain hotspots",
            "operations, supply chain and financed/insured activities"
        ])
        row["transition_plan_key_levers"] = pick_one([
            "energy efficiency; renewable electricity; fleet/process electrification",
            "portfolio shift; supplier engagement; low-carbon technology deployment",
            "fuel switch; capex alignment; product/service redesign"
        ])
        row["transition_plan_capex_alignment_pct"] = round(np.random.uniform(20, 85), 1)
        row["transition_plan_governance_owner"] = pick_one([
            "CFO and sustainability committee",
            "executive committee",
            "risk committee and business unit leadership",
            "board-level sustainability committee"
        ])
        row["transition_plan_progress_status"] = pick_one([
            "early implementation",
            "on track",
            "partially on track",
            "under review"
        ])
    else:
        row["transition_plan_horizon"] = np.nan
        row["transition_plan_coverage_scope"] = np.nan
        row["transition_plan_key_levers"] = np.nan
        row["transition_plan_capex_alignment_pct"] = np.nan
        row["transition_plan_governance_owner"] = np.nan
        row["transition_plan_progress_status"] = np.nan

    return row

In [73]:
df = df.apply(enrich_ifrs_s2_disclosures, axis=1)

In [74]:
check_cols = [
    "scope1_emissions_tco2e",
    "scope2_emissions_tco2e",
    "scope3_emissions_tco2e",
    "internal_carbon_price",
    "ghg_measurement_standard",
    "emission_factor_source",
    "scope3_category_coverage_pct",
    "climate_risk_likelihood_method",
    "climate_risk_magnitude_method",
    "transition_plan_exists"
]

df[check_cols].isna().sum()

scope1_emissions_tco2e            0
scope2_emissions_tco2e            0
scope3_emissions_tco2e            0
internal_carbon_price             1
ghg_measurement_standard          0
emission_factor_source            0
scope3_category_coverage_pct      0
climate_risk_likelihood_method    0
climate_risk_magnitude_method     0
transition_plan_exists            0
dtype: int64

In [75]:
df["scope3_sum_check"] = df[SCOPE3_CATEGORY_COLUMNS].sum(axis=1).round(2)
df["scope3_reconciliation_gap"] = (df["scope3_emissions_tco2e"] - df["scope3_sum_check"]).round(2)

df[["company_name", "reporting_year", "scope3_emissions_tco2e", "scope3_sum_check", "scope3_reconciliation_gap"]].head(10)

,company_name,reporting_year,scope3_emissions_tco2e,scope3_sum_check,scope3_reconciliation_gap
0,North Africa Transition Bank,2025,4924538.99,4924538.99,0.0
1,Med Solar Utility,2025,184169.37,184169.37,0.0
2,TransMaghreb Logistics,2025,1274728.15,1274728.15,0.0
3,Sahel Protection Insurance,2025,177337.07,177337.07,0.0


In [76]:
df.to_csv("synthetic_ifrs_dataset_v3_enriched.csv", index=False)
print("Saved: synthetic_ifrs_dataset_v3_enriched.csv")

Saved: synthetic_ifrs_dataset_v3_enriched.csv
